# MarkWeightX on a Colab GPU

Learn a watermark into Llama-3.2-1B with your own dataset, from the MarkWeightX run page.

1. **Runtime → Change runtime type → T4 GPU** (or better), then **Runtime → Run all**.
2. Open the link the second cell prints. The page is already connected to this notebook.
3. Upload a dataset (or use the C4 sample), pick a method and its hyperparameters, press **Start**.

A 500-step PRO run on 1B takes a few minutes and about 9 GB of GPU memory. Everything lives in this Colab session and is
deleted when it ends: the last cell downloads a release and its key.

In [ ]:
# 1. Install (about 2 minutes)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install "markweightx @ git+https://github.com/jqxue1999/MarkWeightX"
# a restarted session keeps the installed package: always take the latest code from GitHub
!pip -q install --force-reinstall --no-deps "markweightx @ git+https://github.com/jqxue1999/MarkWeightX"
# Colab ships torchao 0.10; recent peft refuses to build LoRA layers next to a torchao older than 0.16, and
# MarkWeightX does not use torchao
!pip -q uninstall -y torchao
# upstream MarkLLM at the pinned commit: the teachers of the markllm-*-d methods (PRO does not need it)
!git clone -q https://github.com/THU-BPM/MarkLLM.git ~/.markweightx/MarkLLM 2>/dev/null; git -C ~/.markweightx/MarkLLM checkout -q 112fa74
import nltk
for p in ("wordnet", "omw-1.4", "punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"):
    nltk.download(p, quiet=True)

In [ ]:
# 2. Start the server and open the run page
import secrets, socket, subprocess, time
from IPython.display import HTML, display
from google.colab.output import eval_js

PORT, TOKEN = 7860, secrets.token_urlsafe(16)
server = subprocess.Popen(["markweightx", "serve", "--port", str(PORT), "--token", TOKEN],
                          stdout=open("/content/serve.log", "w"), stderr=subprocess.STDOUT)
for _ in range(120):
    try:
        socket.create_connection(("127.0.0.1", PORT), timeout=1).close(); break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError(open("/content/serve.log").read())
url = eval_js(f"google.colab.kernel.proxyPort({PORT})").rstrip("/") + f"/#token={TOKEN}"
display(HTML(f'<a href="{url}" target="_blank" style="font-size:16px">Open the MarkWeightX run page</a>'
             f'<p style="color:#64748b">This link only works in this browser while the notebook runs.</p>'))

## Keep your results

**Runtime → Run all** reaches this cell before any job has run, so it only says so. Run it again after a job is done: it downloads the newest release (the LoRA adapter, `release.json` and the training log)
and the key store. Keep the key private: whoever has it can detect the watermark, and also remove or forge it.

In [ ]:
# 3. Download the newest finished release and the key store (run this cell again after a job is done)
import glob, json, os, shutil
from google.colab import files
done = sorted(d for d in glob.glob(os.path.expanduser("~/.markweightx/serve/jobs/*"))
              if json.load(open(f"{d}/status.json")).get("phase") == "done")
if not done:
    print("No finished job yet. Start one on the run page, and run this cell again when it says done.")
else:
    shutil.make_archive("/content/release", "zip", f"{done[-1]}/release")
    shutil.make_archive("/content/keys", "zip", os.path.expanduser("~/.markweightx/runs/keys"))
    print("release of job", os.path.basename(done[-1]))
    files.download("/content/release.zip"); files.download("/content/keys.zip")